# Skyntri — Model B: Skin Disease Screening Support (EfficientNet, face + body)

This notebook trains the model behind **Skin Analysis → "Skin disease screening"**. It is written for **Google Colab** with a T4 GPU. **It has not been run.**

> **Safety, built in**
> * *Suspicious lesion* (melanoma, basal cell carcinoma, squamous cell carcinoma, actinic keratosis) is a **referral flag**. Its threshold is tuned for **high sensitivity**, and it is checked **before** any other answer.
> * *Benign-appearing lesion* is never shown as "harmless". The app tells the user that only a doctor can confirm that.
> * Low confidence → the app asks for another photo.
> * The backend keeps Model B **off** until you set `DISEASE_SCREENING_ENABLED=true`.

## Datasets (research summary and links)

| Role | Dataset | Link | Why |
|---|---|---|---|
| **Primary** | **Skin Diseases Image Dataset** (Ismail Hossain, Kaggle), ~27k images, 10 classes | https://www.kaggle.com/datasets/ismailpromus/skin-diseases-image-dataset | Largest ready-made set covering eczema, atopic dermatitis, psoriasis/lichen planus, fungal and viral infections, and lesions (melanoma, BCC, nevi, keratoses) |
| Phone photos of lesions | **PAD-UFES-20**, 2,298 smartphone images, CC BY 4.0 | https://data.mendeley.com/datasets/zr7vgbcyr2/1 | Smartphone photos (like the app) of BCC, SCC, melanoma, actinic keratosis, nevus, seborrheic keratosis |
| Optional: more lesions | **ISIC 2019**, 25,331 dermoscopy images, CC BY-NC | https://challenge.isic-archive.com/data/#2019 (images: https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip, labels: https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv) | More melanoma, BCC and SCC examples from the ISIC Archive |
| **External test only** | **DDI** (Stanford), biopsy-proven, all skin tones | https://aimi.stanford.edu/datasets/ddi-diverse-dermatology-images | Independent fairness test; never trained on |

**Known limitation:** the lesion classes in the primary dataset and in ISIC are mostly **dermoscopy** images (taken with a special magnifier), not phone photos. PAD-UFES-20 is added so the model also sees phone photos, and the evaluation reports results **separately for phone and dermoscopy images**.

Check each licence; all of these allow education and research use.

## Classes used by the app

| App class | Folders / labels that map to it |
|---|---|
| Eczema / Atopic dermatitis | Eczema, Atopic Dermatitis |
| Psoriasis / Lichen planus | Psoriasis pictures Lichen Planus and related diseases |
| Fungal infection | Tinea Ringworm Candidiasis and other Fungal Infections |
| Viral infection (warts / molluscum) | Warts Molluscum and other Viral Infections |
| Benign-appearing lesion | Melanocytic Nevi, Benign Keratosis-like Lesions, Seborrheic Keratoses, NEV, SEK, DF, VASC |
| **Suspicious lesion** (referral) | Melanoma, Basal Cell Carcinoma, MEL, BCC, SCC, AK/ACK |

## 0. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab already has TensorFlow and every other library these notebooks use, so nothing is installed here.
# Installing another TensorFlow version only causes dependency conflicts. The exported .keras files load
# in the backend, which uses its own TensorFlow (backend/requirements.txt).
if not IN_COLAB:
    print("Outside Colab, install first: pip install tensorflow opencv-python-headless scikit-learn pandas matplotlib seaborn tqdm kaggle")

In [ ]:
import hashlib
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================== CONFIGURATION ==============================
SEED = 42
DATA_ROOT = Path("/content/drive/MyDrive/skyntri/datasets/disease")
WORK_DIR = Path("/content/drive/MyDrive/skyntri/work/disease")
EXPORT_DIR = Path("/content/drive/MyDrive/skyntri/export")

PRIMARY_ROOT = DATA_ROOT / "skin_diseases_10class"   # Kaggle ismailpromus/skin-diseases-image-dataset (unzipped)
PAD_ROOT = DATA_ROOT / "pad_ufes_20"                 # metadata.csv + images (any sub-folders)
ISIC2019_ROOT = DATA_ROOT / "isic2019"               # ISIC_2019_Training_GroundTruth.csv + ISIC_2019_Training_Input/
DDI_ROOT = DATA_ROOT / "ddi"                         # ddi_metadata.csv + images (EXTERNAL TEST ONLY)
USE_ISIC2019 = False                                 # True = add ISIC 2019 lesions (more data, more dermoscopy)

BACKBONE = "EfficientNetB3"
IMG_SIZE = {"EfficientNetB0": 224, "EfficientNetB3": 300, "EfficientNetV2S": 384}[BACKBONE]
PRETRAINED_WEIGHTS = "imagenet"
BATCH_SIZE = 32
HEAD_EPOCHS = 6
FINE_TUNE_EPOCHS = 30
HEAD_LR = 1e-3
FINE_TUNE_LR = 4e-5
UNFREEZE_FRACTION = 0.6
LABEL_SMOOTHING = 0.1
MIXED_PRECISION = True

MODEL_VERSION = f"skin-disease-{BACKBONE.lower()}-v2"
DATASET_VERSION = time.strftime("%Y-%m-%d")

MIN_SIDE = 96
NEAR_DUP_HAMMING = 4
CACHE_SIDE = 448
MAX_IMAGES_PER_CLASS = 4000      # stops nevi/melanoma (with many augmented copies) dominating
TARGET_REFERRAL_SENSITIVITY = 0.95
TARGET_SELECTIVE_ACCURACY = 0.80
MIN_COVERAGE = 0.75

CLASSES = [
    "Eczema / Atopic dermatitis",
    "Psoriasis / Lichen planus",
    "Fungal infection",
    "Viral infection (warts / molluscum)",
    "Benign-appearing lesion",
    "Suspicious lesion",
]
REFERRAL_CLASSES = ["Suspicious lesion"]
CAUTION_CLASSES = ["Benign-appearing lesion"]
NUM_CLASSES = len(CLASSES)
CLASS_INDEX = {c: i for i, c in enumerate(CLASSES)}

# keyword -> class; checked in order, first match wins ("suspicious" before "benign" on purpose)
TAXONOMY_RULES = [
    ("Suspicious lesion", ["melanoma", "basal cell", "bcc", "squamous", "scc", "carcinoma", "actinic", "akiec"]),
    ("Benign-appearing lesion", ["nevi", "nevus", "benign keratosis", "bkl", "seborrheic", "keratoses",
                                 "dermatofibroma", "vascular"]),
    ("Psoriasis / Lichen planus", ["psoriasis", "lichen"]),
    ("Fungal infection", ["tinea", "ringworm", "candid", "fungal"]),
    ("Viral infection (warts / molluscum)", ["wart", "molluscum", "viral"]),
    ("Eczema / Atopic dermatitis", ["eczema", "atopic", "dermatitis"]),
]
PAD_CODES = {"MEL": "Suspicious lesion", "BCC": "Suspicious lesion", "SCC": "Suspicious lesion",
             "ACK": "Suspicious lesion", "NEV": "Benign-appearing lesion", "SEK": "Benign-appearing lesion"}
ISIC_CODES = {"MEL": "Suspicious lesion", "BCC": "Suspicious lesion", "SCC": "Suspicious lesion",
              "AK": "Suspicious lesion", "NV": "Benign-appearing lesion", "BKL": "Benign-appearing lesion",
              "DF": "Benign-appearing lesion", "VASC": "Benign-appearing lesion"}
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
for p in (WORK_DIR, EXPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)
if MIXED_PRECISION and tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")


def map_text(text):
    t = str(text).lower().replace("_", " ")
    for cls, keys in TAXONOMY_RULES:
        if any(k in t for k in keys):
            return cls
    return None

## 1. Get the data

* **Primary (Kaggle API):** upload your `kaggle.json` when asked, or download it manually and unzip it into `DATA_ROOT/skin_diseases_10class/`.
* **PAD-UFES-20:** the cell below downloads it from Mendeley and copies `metadata.csv` and the images into `DATA_ROOT/pad_ufes_20/`. If the download fails, open https://data.mendeley.com/datasets/zr7vgbcyr2/1, click *Download All*, upload the zip to `DATA_ROOT/pad_ufes_20.zip` (Google Drive) and run the cell again.
* **ISIC 2019** (optional): the cell downloads it when `USE_ISIC2019 = True`. It is about 9 GB.
* **DDI** (optional, external test): request access on the Stanford AIMI page and put it in `DATA_ROOT/ddi/`.

In [ ]:
if IN_COLAB and not (PRIMARY_ROOT.exists() and any(PRIMARY_ROOT.rglob("*.jpg"))):
    from google.colab import files
    print("Upload kaggle.json")
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d ismailpromus/skin-diseases-image-dataset -p "{PRIMARY_ROOT}" --unzip

if USE_ISIC2019 and not (ISIC2019_ROOT / "ISIC_2019_Training_GroundTruth.csv").exists():
    ISIC2019_ROOT.mkdir(parents=True, exist_ok=True)
    !wget -q -O "{ISIC2019_ROOT}/ISIC_2019_Training_GroundTruth.csv" https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv
    !wget -q -O /content/isic2019.zip https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip
    !unzip -q /content/isic2019.zip -d "{ISIC2019_ROOT}"

PAD_ZIP_URL = "https://data.mendeley.com/public-api/zip/zr7vgbcyr2/download/1"
if IN_COLAB and not (PAD_ROOT / "metadata.csv").exists():
    import shutil
    import zipfile

    pad_zip = DATA_ROOT / "pad_ufes_20.zip"          # a zip you downloaded by hand also works
    if not pad_zip.exists():
        !wget -q --show-progress -O /content/pad_ufes_20.zip "{PAD_ZIP_URL}"
        pad_zip = Path("/content/pad_ufes_20.zip")
    if zipfile.is_zipfile(pad_zip):
        tmp = Path("/content/pad_tmp")
        with zipfile.ZipFile(pad_zip) as z:
            z.extractall(tmp)
        for inner in list(tmp.rglob("*.zip")):       # the images come as a zip inside the zip
            with zipfile.ZipFile(inner) as z:
                z.extractall(inner.parent)
        PAD_ROOT.mkdir(parents=True, exist_ok=True)
        meta_files = list(tmp.rglob("metadata.csv"))
        if meta_files:
            shutil.copy(meta_files[0], PAD_ROOT / "metadata.csv")
        (PAD_ROOT / "images").mkdir(exist_ok=True)
        for p in tqdm(list(tmp.rglob("*.png")), desc="PAD-UFES-20 -> Drive"):
            shutil.copy(p, PAD_ROOT / "images" / p.name)
    else:
        print("PAD-UFES-20 download failed. Open https://data.mendeley.com/datasets/zr7vgbcyr2/1, click 'Download All',",
              "upload the zip to", DATA_ROOT / "pad_ufes_20.zip", "and run this cell again.")

for name, root in [("skin_diseases_10class", PRIMARY_ROOT), ("pad_ufes_20", PAD_ROOT), ("isic2019", ISIC2019_ROOT), ("ddi", DDI_ROOT)]:
    n = sum(1 for p in root.rglob("*") if p.suffix.lower() in IMG_EXT) if root.exists() else 0
    print(f"{name:22s} {n:6d} images   {root}")
print("PAD-UFES-20 metadata.csv found:", (PAD_ROOT / "metadata.csv").exists())


## 2. Load all sources into one manifest

In [ ]:
def load_primary():
    rows, skipped = [], defaultdict(int)
    if not PRIMARY_ROOT.exists():
        print("Primary dataset not found:", PRIMARY_ROOT); return rows
    for p in PRIMARY_ROOT.rglob("*"):
        if p.suffix.lower() not in IMG_EXT:
            continue
        folder = " / ".join(p.relative_to(PRIMARY_ROOT).parts[:-1])
        cls = map_text(folder)
        if cls is None:
            skipped[folder] += 1
            continue
        # HAM10000/ISIC-derived lesion folders are dermoscopy; the rest are clinical photos
        dermo = any(k in folder.lower() for k in ["nevi", "melanoma", "basal cell", "benign keratosis"])
        rows.append({"path": str(p), "source": "skin_diseases_10class", "condition": folder, "label": cls,
                     "group": f"prim:{p.stem.split('_jpg.rf.')[0].lower()}",
                     "domain": "dermoscopy" if dermo else "clinical", "fst": "unknown"})
    if skipped:
        print("Primary: skipped folders:", dict(skipped))
    return rows


def fst_group(v):
    try:
        v = int(float(v))
    except (TypeError, ValueError):
        return "unknown"
    return {1: "I-II", 2: "I-II", 3: "III-IV", 4: "III-IV", 5: "V-VI", 6: "V-VI"}.get(v, "unknown")


def load_pad():
    meta = PAD_ROOT / "metadata.csv"
    if not meta.exists():
        print("PAD-UFES-20 not found - skipped"); return []
    df = pd.read_csv(meta)
    index = {p.name: p for p in PAD_ROOT.rglob("*.png")}
    rows = []
    for _, r in df.iterrows():
        p = index.get(str(r["img_id"]))
        cls = PAD_CODES.get(str(r["diagnostic"]))
        if p is None or cls is None:
            continue
        rows.append({"path": str(p), "source": "pad_ufes_20", "condition": r["diagnostic"], "label": cls,
                     "group": f"pad:{r['patient_id']}", "domain": "phone", "fst": fst_group(r.get("fitspatrick"))})
    return rows


def load_isic2019():
    gt = ISIC2019_ROOT / "ISIC_2019_Training_GroundTruth.csv"
    if not (USE_ISIC2019 and gt.exists()):
        return []
    df = pd.read_csv(gt)
    index = {p.stem: p for p in ISIC2019_ROOT.rglob("*.jpg")}
    rows = []
    for _, r in df.iterrows():
        code = max(ISIC_CODES, key=lambda c: r.get(c, 0))
        if r.get(code, 0) < 1 or r["image"] not in index:
            continue
        rows.append({"path": str(index[r["image"]]), "source": "isic2019", "condition": code,
                     "label": ISIC_CODES[code], "group": f"isic:{r['image']}", "domain": "dermoscopy", "fst": "unknown"})
    return rows


manifest = pd.DataFrame(load_primary() + load_pad() + load_isic2019())
display(manifest.groupby(["label", "source"]).size().unstack(fill_value=0))

# Check: every folder / diagnosis code and the class it was given (fix TAXONOMY_RULES if one is wrong)
print(manifest.groupby(["source", "condition", "label"]).size().to_string())

## 3. Cleaning

In [ ]:
def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def inspect(path):
    try:
        with Image.open(path) as im:
            rgb = np.asarray(ImageOps.exif_transpose(im).convert("RGB"))
    except Exception as exc:
        return {"ok": False, "reason": f"unreadable:{type(exc).__name__}"}
    if min(rgb.shape[:2]) < MIN_SIDE:
        return {"ok": False, "reason": "too_small"}
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    small = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA)
    g = cv2.resize(gray, (256, 256))
    return {"ok": True, "reason": None, "md5": file_md5(path),
            "dhash": np.packbits((small[:, 1:] > small[:, :-1]).flatten()).tobytes().hex(),
            "luma": float(g.mean()), "sharpness": float(cv2.Laplacian(g, cv2.CV_64F).var())}


info = pd.DataFrame([inspect(p) for p in tqdm(manifest["path"], desc="inspecting")])
manifest = pd.concat([manifest.reset_index(drop=True), info], axis=1)
print(manifest["reason"].value_counts())
manifest = manifest[manifest["ok"] == True].copy()  # noqa: E712
bad = (manifest.luma < 20) | (manifest.luma > 245) | (manifest.sharpness < 4)
print("Removed for quality:", int(bad.sum()))
manifest = manifest[~bad]

conflict = manifest.groupby("md5")["label"].nunique()
conflict = set(conflict[conflict > 1].index)
before = len(manifest)
manifest = manifest[~manifest.md5.isin(conflict)].drop_duplicates("md5").reset_index(drop=True)
print(f"Exact duplicates removed: {before - len(manifest)} (conflicting-label hashes: {len(conflict)})")

# Near duplicates (the primary dataset contains augmented copies) -> one group, then keep one image
parent = {g: g for g in manifest["group"]}


def find(g):
    while parent[g] != g:
        parent[g] = parent[parent[g]]
        g = parent[g]
    return g


hashes = np.array([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in manifest["dhash"]])
bits = np.unpackbits(hashes, axis=1)
groups = manifest["group"].tolist()
for band in range(4):
    buckets = defaultdict(list)
    for i, row in enumerate(hashes):
        buckets[bytes(row[band * 2: band * 2 + 2])].append(i)
    for idx in buckets.values():
        sub = bits[idx]
        for a in range(len(idx) - 1):
            dist = (sub[a + 1:] != sub[a]).sum(1)
            for off in np.nonzero(dist <= NEAR_DUP_HAMMING)[0]:
                ra, rb = find(groups[idx[a]]), find(groups[idx[a + 1 + off]])
                if ra != rb:
                    parent[rb] = ra
manifest["group"] = [find(g) for g in groups]
mixed = manifest.groupby("group")["label"].nunique()
mixed = set(mixed[mixed > 1].index)
manifest = manifest[~manifest.group.isin(mixed)]
# Patients (PAD) keep all their images (different lesions); other sources keep one image per near-dup group
pad = manifest[manifest.source == "pad_ufes_20"]
other = manifest[manifest.source != "pad_ufes_20"].sort_values("sharpness", ascending=False).drop_duplicates("group")
manifest = pd.concat([pad, other]).reset_index(drop=True)
print("Images after de-duplication:", len(manifest), "| groups:", manifest.group.nunique())

## 4. Cache images and balance classes

In [ ]:
CACHE_DIR = WORK_DIR / f"cleaned_{CACHE_SIDE}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_image(row):
    out = CACHE_DIR / f"{row['md5']}.jpg"
    if not out.exists():
        with Image.open(row["path"]) as im:
            img = ImageOps.exif_transpose(im).convert("RGB")
            img.thumbnail((CACHE_SIDE, CACHE_SIDE), Image.Resampling.LANCZOS)
            img.save(out, quality=95)
    return str(out)


manifest["clean_path"] = [cache_image(r) for r in tqdm(manifest.to_dict("records"), desc="caching")]
# Cap classes; always keep every phone (PAD) image because they match the app best
capped = []
for cls, d in manifest.groupby("label"):
    phone, rest = d[d.domain == "phone"], d[d.domain != "phone"]
    room = max(0, MAX_IMAGES_PER_CLASS - len(phone))
    capped.append(pd.concat([phone, rest.sample(min(len(rest), room), random_state=SEED)]))
manifest = pd.concat(capped).reset_index(drop=True)
manifest = manifest[manifest.label.isin(CLASSES)].reset_index(drop=True)
manifest["y"] = manifest["label"].map(CLASS_INDEX)
manifest.to_pickle(WORK_DIR / "manifest_clean.pkl")

display(manifest.groupby(["label", "domain"]).size().unstack(fill_value=0))
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(12, 3 * NUM_CLASSES))
for r, cls in enumerate(CLASSES):
    s = manifest[manifest.label == cls]
    s = s.sample(min(4, len(s)), random_state=SEED)
    for c in range(4):
        axes[r, c].axis("off")
        if c < len(s):
            axes[r, c].imshow(Image.open(s.iloc[c]["clean_path"]))
            axes[r, c].set_title(f"{cls}\n{s.iloc[c]['domain']}", fontsize=8)
plt.tight_layout(); plt.show()

## 5. Grouped, stratified split (a patient or a near-duplicate group never crosses splits)

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = list(sgkf.split(manifest, manifest["y"], manifest["group"]))
manifest["split"] = "train"
manifest.loc[folds[0][1], "split"] = "test"
manifest.loc[folds[1][1], "split"] = "val"
assert manifest.groupby("group")["split"].nunique().max() == 1
display(manifest.groupby(["label", "split"]).size().unstack(fill_value=0))
manifest.to_csv(WORK_DIR / "manifest_split.csv", index=False)
train_df = manifest[manifest.split == "train"].reset_index(drop=True)
val_df = manifest[manifest.split == "val"].reset_index(drop=True)
test_df = manifest[manifest.split == "test"].reset_index(drop=True)

## 6. Pipeline and augmentation (body photos come in any orientation; colour changes kept small)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2, fill_mode="reflect"),
    tf.keras.layers.RandomZoom((-0.2, 0.15), fill_mode="reflect"),
    tf.keras.layers.RandomTranslation(0.08, 0.08, fill_mode="reflect"),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomBrightness(0.10, value_range=(0, 255)),
], name="augment")


def load_image(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.cast(tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear"), tf.float32)


def make_ds(df, training):
    ds = tf.data.Dataset.from_tensor_slices((df["clean_path"].values, tf.one_hot(df["y"].values, NUM_CLASSES)))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0, 255), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)

## 7. Model

In [ ]:
present = np.unique(train_df["y"].values)
cw = compute_class_weight("balanced", classes=present, y=train_df["y"].values)
class_weights = {int(c): float(min(w, 8.0)) for c, w in zip(present, cw)}
# extra weight on the referral class: missing a suspicious lesion is the worst error
for c in REFERRAL_CLASSES:
    if CLASS_INDEX[c] in class_weights:
        class_weights[CLASS_INDEX[c]] *= 1.5
print({CLASSES[i]: round(w, 2) for i, w in class_weights.items()})


def build_model(weights=PRETRAINED_WEIGHTS):
    base = getattr(tf.keras.applications, BACKBONE)(include_top=False, weights=weights,
                                                     input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="conditions")(x)
    return tf.keras.Model(inputs, outputs, name=f"skyntri_disease_{BACKBONE.lower()}"), base


def predict_probs(m, ds, tta=False):
    out = []
    for x, _ in ds:
        p = m.predict_on_batch(x)
        if tta:
            p = (p + m.predict_on_batch(tf.reverse(x, axis=[2]))) / 2.0
        out.append(p)
    return np.concatenate(out, axis=0)


class ValMetrics(tf.keras.callbacks.Callback):
    """Balanced accuracy + referral recall each epoch (checkpoint on their mean)."""

    def __init__(self, ds, y):
        super().__init__()
        self.ds, self.y = ds, y

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        pred = predict_probs(self.model, self.ds).argmax(1)
        bal = balanced_accuracy_score(self.y, pred)
        s = CLASS_INDEX["Suspicious lesion"]
        rec = ((pred == s) & (self.y == s)).sum() / max((self.y == s).sum(), 1)
        logs["val_bal_acc"], logs["val_referral_recall"] = bal, rec
        logs["val_score"] = (bal + rec) / 2
        print(f" - val_bal_acc {bal:.4f} - val_referral_recall {rec:.4f}")


model, base = build_model()
LOSS = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
METRICS = [tf.keras.metrics.CategoricalAccuracy(name="acc"), tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2")]
CKPT = WORK_DIR / "best_disease.keras"


def callbacks(patience):
    return [
        ValMetrics(val_ds, val_df["y"].values),
        tf.keras.callbacks.ModelCheckpoint(str(CKPT), monitor="val_score", mode="max", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_score", mode="max", patience=patience, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(str(WORK_DIR / "training_log.csv"), append=True),
    ]

## 8. Training (run this yourself)

In [ ]:
model.compile(optimizer=tf.keras.optimizers.AdamW(HEAD_LR, weight_decay=1e-4), loss=LOSS, metrics=METRICS)
model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=class_weights, callbacks=callbacks(3))

base.trainable = True
cut = int(len(base.layers) * (1 - UNFREEZE_FRACTION))
for i, layer in enumerate(base.layers):
    layer.trainable = i >= cut and not isinstance(layer, tf.keras.layers.BatchNormalization)
steps = FINE_TUNE_EPOCHS * max(1, math.ceil(len(train_df) / BATCH_SIZE))
model.compile(optimizer=tf.keras.optimizers.AdamW(
    tf.keras.optimizers.schedules.CosineDecay(FINE_TUNE_LR, steps, alpha=0.05), weight_decay=1e-5),
    loss=LOSS, metrics=METRICS)
model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, class_weight=class_weights, callbacks=callbacks(7))

log = pd.read_csv(WORK_DIR / "training_log.csv")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
log[["loss", "val_loss"]].plot(ax=a1, title="Loss")
log[["val_bal_acc", "val_referral_recall"]].plot(ax=a2, title="Validation")
plt.show()

## 9. Operating points (validation only)

* **`referral_threshold`:** the largest `P(Suspicious lesion)` that still catches `TARGET_REFERRAL_SENSITIVITY` (95%) of suspicious lesions. The flag is raised whenever the probability reaches it, **even if another class wins**.
* **`abstain_below`:** below this top probability, the app asks for another photo.

In [ ]:
best = tf.keras.models.load_model(CKPT, compile=False)
USE_TTA = True
p_val, y_val = predict_probs(best, val_ds, tta=USE_TTA), val_df["y"].values
s = CLASS_INDEX["Suspicious lesion"]
is_s = (y_val == s).astype(int)
if 0 < is_s.sum() < len(is_s):
    fpr, tpr, thr = roc_curve(is_s, p_val[:, s])
    ok = np.where(tpr >= TARGET_REFERRAL_SENSITIVITY)[0]
    referral_threshold = float(np.clip(thr[ok[0]], 0.01, 0.95)) if len(ok) else 0.05
else:
    referral_threshold = 0.05

conf, correct = p_val.max(1), p_val.argmax(1) == y_val
abstain_below = 0.0
for t in np.linspace(0.2, 0.9, 71):
    kept = conf >= t
    if kept.mean() < MIN_COVERAGE:
        break
    if correct[kept].mean() >= TARGET_SELECTIVE_ACCURACY:
        abstain_below = float(t)
        break
print("referral_threshold:", round(referral_threshold, 4), "| abstain_below:", round(abstain_below, 3))

## 10. Test evaluation

In [ ]:
p_test, y_test = predict_probs(best, test_ds, tta=USE_TTA), test_df["y"].values
y_pred = p_test.argmax(1)
is_s = y_test == s
flag = p_test[:, s] >= referral_threshold

rows = []
for k, cls in enumerate(CLASSES):
    t, p = y_test == k, y_pred == k
    tp, fn, fp, tn = (t & p).sum(), (t & ~p).sum(), (~t & p).sum(), (~t & ~p).sum()
    rows.append({"class": cls, "support": int(t.sum()), "sensitivity": tp / max(tp + fn, 1),
                 "specificity": tn / max(tn + fp, 1), "PPV": tp / max(tp + fp, 1), "NPV": tn / max(tn + fn, 1),
                 "f1": f1_score(t, p, zero_division=0),
                 "roc_auc": roc_auc_score(t, p_test[:, k]) if 0 < t.sum() < len(t) else np.nan})
per_class = pd.DataFrame(rows).set_index("class")
display(per_class.round(3))

summary = {
    "test_images": int(len(y_test)),
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "top2_accuracy": top_k_accuracy_score(y_test, p_test, k=2, labels=np.arange(NUM_CLASSES)),
    "macro_roc_auc_ovr": float(np.nanmean(per_class["roc_auc"])),
    "referral_sensitivity": float((flag & is_s).sum() / max(is_s.sum(), 1)),
    "referral_specificity": float((~flag & ~is_s).sum() / max((~is_s).sum(), 1)),
}
kept = p_test.max(1) >= abstain_below
summary["coverage_after_abstention"] = float(kept.mean())
summary["accuracy_on_answered"] = float((y_pred[kept] == y_test[kept]).mean()) if kept.any() else float("nan")
print(classification_report(y_test, y_pred, labels=np.arange(NUM_CLASSES), target_names=CLASSES, zero_division=0))
pd.Series(summary).round(4)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=np.arange(NUM_CLASSES), normalize="true")
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2f", xticklabels=CLASSES, yticklabels=CLASSES, cmap="Blues")
plt.xlabel("predicted"); plt.ylabel("true"); plt.xticks(rotation=35, ha="right"); plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 6))
for k, cls in enumerate(CLASSES):
    t = (y_test == k).astype(int)
    if 0 < t.sum() < len(t):
        f, tp_, _ = roc_curve(t, p_test[:, k]); plt.plot(f, tp_, label=f"{cls} ({roc_auc_score(t, p_test[:, k]):.2f})")
plt.plot([0, 1], [0, 1], "k--"); plt.legend(fontsize=7); plt.title("One-vs-rest ROC"); plt.show()


def ece(conf, correct, bins=10):
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)


summary["ece_top1"] = ece(p_test.max(1), (y_pred == y_test).astype(float))
print("ECE:", round(summary["ece_top1"], 4))

In [ ]:
def bootstrap(fn, n=1000):
    rng = np.random.default_rng(SEED)
    gidx = test_df.groupby("group").indices
    keys = list(gidx)
    vals = [fn(np.concatenate([gidx[k] for k in rng.choice(keys, len(keys), replace=True)])) for _ in range(n)]
    return [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))]


ci = {
    "balanced_accuracy": bootstrap(lambda i: balanced_accuracy_score(y_test[i], y_pred[i])),
    "macro_f1": bootstrap(lambda i: f1_score(y_test[i], y_pred[i], average="macro", zero_division=0)),
    "referral_sensitivity": bootstrap(lambda i: (flag[i] & is_s[i]).sum() / max(is_s[i].sum(), 1)),
    "referral_specificity": bootstrap(lambda i: (~flag[i] & ~is_s[i]).sum() / max((~is_s[i]).sum(), 1)),
}
for k, v in ci.items():
    print(f"{k}: {summary[k]:.3f}  95% CI {v[0]:.3f}-{v[1]:.3f}")

# Results by image type (phone vs clinical vs dermoscopy) and by Fitzpatrick group (PAD only)
by_domain = pd.DataFrame([
    {"domain": d, "n": len(i), "balanced_accuracy": balanced_accuracy_score(y_test[i], y_pred[i]),
     "referral_sensitivity": (flag[i] & is_s[i]).sum() / is_s[i].sum() if is_s[i].sum() else np.nan}
    for d, i in test_df.groupby("domain").indices.items()])
display(by_domain)
fairness = pd.DataFrame([
    {"fitzpatrick": g, "n": len(i), "referral_sensitivity": (flag[i] & is_s[i]).sum() / is_s[i].sum() if is_s[i].sum() else np.nan}
    for g, i in test_df.groupby("fst").indices.items() if g != "unknown"])
display(fairness)

In [ ]:
# Missed suspicious lesions -> must be reviewed by a clinician before release
fn_idx = np.where(is_s & ~flag)[0]
print("Missed suspicious lesions:", len(fn_idx))
if len(fn_idx):
    fig, axes = plt.subplots(1, min(8, len(fn_idx)), figsize=(3 * min(8, len(fn_idx)), 3.5))
    for ax, i in zip(np.atleast_1d(axes), fn_idx[:8]):
        ax.imshow(Image.open(test_df.loc[i, "clean_path"])); ax.axis("off")
        ax.set_title(f"{test_df.loc[i, 'condition']}\nP={p_test[i, s]:.2f}", fontsize=8)
    plt.show()
test_df.loc[fn_idx, ["clean_path", "condition", "source", "domain"]].to_csv(WORK_DIR / "missed_suspicious_for_review.csv", index=False)

# External test on DDI (never used for training or thresholds)
external = None
if (DDI_ROOT / "ddi_metadata.csv").exists():
    ddi = pd.read_csv(DDI_ROOT / "ddi_metadata.csv")
    index = {p.name: p for p in DDI_ROOT.rglob("*.png")}
    ddi["path"] = ddi["DDI_file"].map(lambda f: str(index.get(f, "")))
    ddi = ddi[ddi.path != ""].reset_index(drop=True)
    ds = tf.data.Dataset.from_tensor_slices(ddi["path"].values).map(load_image).batch(BATCH_SIZE)
    p_ddi = np.concatenate([best.predict_on_batch(x) for x in ds])
    mal = ddi["malignant"].astype(str).str.lower().isin(["true", "1"]).values
    f_ddi = p_ddi[:, s] >= referral_threshold
    external = pd.DataFrame([{"skin_tone": t, "n": len(i), "malignant": int(mal[i].sum()),
                              "sensitivity": (f_ddi[i] & mal[i]).sum() / max(mal[i].sum(), 1),
                              "specificity": (~f_ddi[i] & ~mal[i]).sum() / max((~mal[i]).sum(), 1)}
                             for t, i in ddi.groupby("skin_tone").indices.items()])
    display(external)
else:
    print("DDI not provided - external test skipped (recommended before release).")

## 11. Export for the Skyntri backend

Copy **both files** into **`backend/models/`**:

| File | Loaded by |
|---|---|
| `skin_disease_efficientnet.keras` | `backend/app/services/disease_screening.py` → `load_model()` (setting `DISEASE_MODEL_PATH`) |
| `skin_disease_metadata.json` | `backend/app/services/disease_screening.py` → `load_metadata()` (setting `DISEASE_MODEL_METADATA_PATH`) |

Then set `DISEASE_SCREENING_ENABLED=true` in `.env`, but **only after a doctor has reviewed the results**, and restart the backend.

In [ ]:
MODEL_FILE = EXPORT_DIR / "skin_disease_efficientnet.keras"
META_FILE = EXPORT_DIR / "skin_disease_metadata.json"
tf.keras.mixed_precision.set_global_policy("float32")
export_model, _ = build_model()  # ImageNet build = same layers as in training (weights=None drops a Rescaling layer)
export_model.set_weights(best.get_weights())
export_model.save(MODEL_FILE)
sha256 = hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest()

metadata = {
    "task": "disease_screening",
    "model_version": MODEL_VERSION,
    "backbone": BACKBONE,
    "image_size": IMG_SIZE,
    "output": "softmax",
    "preprocessing": "RGB, bilinear resize to image_size, float32 0-255 (no /255)",
    "tta": USE_TTA,
    "classes": CLASSES,
    "referral_classes": REFERRAL_CLASSES,
    "caution_classes": CAUTION_CLASSES,
    "other_class": "Other / not supported",
    "referral_threshold": referral_threshold,
    "abstain_below": abstain_below,
    "sha256": sha256,
    "dataset_version": DATASET_VERSION,
    "sources": sorted(manifest["source"].unique().tolist()),
    "split_sizes": {k: int((manifest.split == k).sum()) for k in ("train", "val", "test")},
    "test_metrics": {k: round(float(v), 4) for k, v in summary.items()},
    "test_metrics_ci95": ci,
    "per_class": per_class.round(4).reset_index().to_dict(orient="records"),
    "by_domain": by_domain.round(4).to_dict(orient="records"),
    "external_ddi": None if external is None else external.round(4).to_dict(orient="records"),
    "clinical_review": "PENDING - keep DISEASE_SCREENING_ENABLED=false until a dermatologist signs off",
}
META_FILE.write_text(json.dumps(metadata, indent=2, default=float))
print("Saved", MODEL_FILE, META_FILE)

## 12. Safe prediction (same rules as the backend)

In [ ]:
loaded = tf.keras.models.load_model(MODEL_FILE, compile=False)
meta = json.loads(META_FILE.read_text())


def predict_disease(path):
    with Image.open(path) as im:
        arr = np.asarray(ImageOps.exif_transpose(im).convert("RGB").resize((meta["image_size"],) * 2,
                                                                            Image.Resampling.BILINEAR), np.float32)
    batch = np.stack([arr, arr[:, ::-1, :]] if meta["tta"] else [arr])
    probs = loaded.predict(batch, verbose=0).mean(axis=0)
    classes = meta["classes"]
    top = classes[int(np.argmax(probs))]
    if any(probs[classes.index(c)] >= meta["referral_threshold"] for c in meta["referral_classes"]):
        status, msg = "referral", "Some features may need a doctor's assessment. Please see a dermatologist soon."
    elif probs.max() < meta["abstain_below"]:
        status, msg = "uncertain", "We cannot confidently assess this image."
    elif top in meta["caution_classes"]:
        status, msg = "lesion", "Features of a skin lesion. Only a doctor can confirm it is harmless."
    else:
        status, msg = "pattern", f"Features may be consistent with {top}. Not a diagnosis."
    return {"status": status, "message": msg, "top_class": top,
            "probabilities": {c: round(float(p), 3) for c, p in zip(classes, probs)}}


xb, _ = next(iter(test_ds))
# `best` runs in mixed precision (float16), the exported copy in float32: small differences are expected
p_train, p_export = np.asarray(best.predict_on_batch(xb), np.float32), np.asarray(loaded.predict_on_batch(xb), np.float32)
same_top = float((p_train.argmax(1) == p_export.argmax(1)).mean())
print(f"export check: max prob difference {np.abs(p_train - p_export).max():.4f}, same top class {same_top:.1%}")
assert same_top >= 0.95 and np.abs(p_train - p_export).max() < 0.05, "exported model differs from the trained one"
predict_disease(test_df.loc[0, "path"])